In [3]:
!pip -q install ultralytics opencv-python pyyaml


In [4]:
from pathlib import Path

PROJECT_DIR = Path(r"C:/Users/HashTag/Desktop/VehicleClassificationYOLO")

DATASET_DIR = PROJECT_DIR / "data" / "vehicles"

IMAGES_TRAIN = DATASET_DIR / "images" / "train"
IMAGES_VAL   = DATASET_DIR / "images" / "val"

JSON_TRAIN = DATASET_DIR / "labels" / "train"
JSON_VAL   = DATASET_DIR / "labels" / "val"

print("DATASET_DIR:", DATASET_DIR)
print("IMAGES_TRAIN exists:", IMAGES_TRAIN.exists())
print("IMAGES_VAL exists:", IMAGES_VAL.exists())
print("JSON_TRAIN exists:", JSON_TRAIN.exists())
print("JSON_VAL exists:", JSON_VAL.exists())


DATASET_DIR: C:\Users\HashTag\Desktop\VehicleClassificationYOLO\data\vehicles
IMAGES_TRAIN exists: True
IMAGES_VAL exists: True
JSON_TRAIN exists: True
JSON_VAL exists: True


In [5]:
CLASS_NAMES = ["car", "bus", "truck", "motorcycle"]

NAME_ALIASES = {
    "motorbike": "motorcycle",
    "bike": "motorcycle",
    "van": "car",
}

name_to_id = {n: i for i, n in enumerate(CLASS_NAMES)}
print("name_to_id:", name_to_id)


name_to_id: {'car': 0, 'bus': 1, 'truck': 2, 'motorcycle': 3}


In [6]:
import json
from collections import defaultdict
import cv2

IMG_EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp")

def find_image_by_stem(img_dir: Path, stem: str):
    for ext in IMG_EXTS:
        p = img_dir / f"{stem}{ext}"
        if p.exists():
            return p
    for p in img_dir.iterdir():
        if p.is_file() and p.stem == stem and p.suffix.lower() in IMG_EXTS:
            return p
    return None

def normalize_name(n: str) -> str:
    n2 = n.strip().lower()
    return NAME_ALIASES.get(n2, n2)

def xyxy_to_yolo(xmin, ymin, xmax, ymax, w, h):
    xmin = max(0, min(xmin, w))
    xmax = max(0, min(xmax, w))
    ymin = max(0, min(ymin, h))
    ymax = max(0, min(ymax, h))
    bw = max(0.0, xmax - xmin)
    bh = max(0.0, ymax - ymin)
    cx = xmin + bw / 2.0
    cy = ymin + bh / 2.0
    return cx / w, cy / h, bw / w, bh / h

def write_yolo_txt(out_path: Path, rows):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        for cls_id, x, y, bw, bh in rows:
            f.write(f"{cls_id} {x:.6f} {y:.6f} {bw:.6f} {bh:.6f}\n")

def detect_json_type(obj: dict):
    if "images" in obj and "annotations" in obj and "categories" in obj:
        return "coco"
    if "shapes" in obj and "imageWidth" in obj and "imageHeight" in obj:
        return "labelme"
    if "bboxes" in obj or "bbox" in obj or "objects" in obj:
        return "generic"
    return "unknown"

def convert_coco(coco: dict, images_dir: Path, out_dir: Path):
    cat_id_to_name = {c["id"]: normalize_name(c.get("name","")) for c in coco["categories"]}
    img_id_to_file = {im["id"]: im.get("file_name") for im in coco["images"]}
    img_id_to_size = {im["id"]: (im.get("width"), im.get("height")) for im in coco["images"]}

    anns_by_img = defaultdict(list)
    for ann in coco["annotations"]:
        anns_by_img[ann["image_id"]].append(ann)

    written = 0
    for img_id, file_name in img_id_to_file.items():
        if not file_name:
            continue
        w, h = img_id_to_size.get(img_id, (None, None))
        if not w or not h:
            img_path = find_image_by_stem(images_dir, Path(file_name).stem)
            if img_path is None:
                continue
            im = cv2.imread(str(img_path))
            h, w = im.shape[:2]

        stem = Path(file_name).stem
        rows = []
        for ann in anns_by_img.get(img_id, []):
            name = cat_id_to_name.get(ann.get("category_id"), "")
            if name not in name_to_id:
                continue
            bbox = ann.get("bbox")  # COCO xywh
            if not bbox or len(bbox) != 4:
                continue
            x, y, bw, bh = bbox
            xmin, ymin, xmax, ymax = x, y, x+bw, y+bh
            xc, yc, ww, hh = xyxy_to_yolo(xmin, ymin, xmax, ymax, w, h)
            rows.append((name_to_id[name], xc, yc, ww, hh))

        if rows:
            write_yolo_txt(out_dir / f"{stem}.txt", rows)
            written += 1
    print("COCO conversion wrote:", written, "txt files")

def convert_labelme(j: dict, json_path: Path, out_dir: Path):
    w, h = j["imageWidth"], j["imageHeight"]
    rows = []
    for sh in j["shapes"]:
        label = normalize_name(sh.get("label",""))
        if label not in name_to_id:
            continue
        pts = sh.get("points", [])
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        xmin, xmax = min(xs), max(xs)
        ymin, ymax = min(ys), max(ys)
        xc, yc, ww, hh = xyxy_to_yolo(xmin, ymin, xmax, ymax, w, h)
        rows.append((name_to_id[label], xc, yc, ww, hh))

    if rows:
        write_yolo_txt(out_dir / f"{json_path.stem}.txt", rows)
        return 1
    return 0

def convert_generic(j: dict, json_path: Path, images_dir: Path, out_dir: Path):
    img_path = find_image_by_stem(images_dir, json_path.stem)
    if img_path is None:
        return 0
    im = cv2.imread(str(img_path))
    h, w = im.shape[:2]

    objs = []
    if isinstance(j.get("bboxes"), list): objs = j["bboxes"]
    elif isinstance(j.get("objects"), list): objs = j["objects"]
    elif isinstance(j.get("bbox"), list): objs = [{"label": j.get("label",""), "bbox": j["bbox"]}]

    rows = []
    for o in objs:
        label = normalize_name(o.get("label") or o.get("class") or "")
        if label not in name_to_id:
            continue
        bbox = o.get("bbox")
        if not bbox or len(bbox) != 4:
            continue

        # guess xyxy first, fallback xywh
        x1,y1,x2,y2 = bbox
        if x2 < x1 or y2 < y1:  # looks like xywh
            x,y,bw,bh = bbox
            xmin,ymin,xmax,ymax = x,y,x+bw,y+bh
        else:
            xmin,ymin,xmax,ymax = x1,y1,x2,y2

        xc,yc,ww,hh = xyxy_to_yolo(xmin,ymin,xmax,ymax,w,h)
        rows.append((name_to_id[label], xc, yc, ww, hh))

    if rows:
        write_yolo_txt(out_dir / f"{json_path.stem}.txt", rows)
        return 1
    return 0

def convert_folder(json_dir: Path, images_dir: Path, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    json_files = sorted(list(json_dir.glob("*.json")))
    if not json_files:
        print("No json files in:", json_dir)
        return

    # COCO single-file case
    if len(json_files) == 1:
        obj = json.loads(json_files[0].read_text(encoding="utf-8"))
        if detect_json_type(obj) == "coco":
            convert_coco(obj, images_dir, out_dir)
            return

    # per-image jsons
    written = 0
    unknown = []
    for jp in json_files:
        obj = json.loads(jp.read_text(encoding="utf-8"))
        t = detect_json_type(obj)
        if t == "labelme":
            written += convert_labelme(obj, jp, out_dir)
        elif t == "generic":
            written += convert_generic(obj, jp, images_dir, out_dir)
        else:
            if len(unknown) < 3:
                unknown.append((jp.name, list(obj.keys())[:25]))
    print("Per-image conversion wrote:", written, "txt files")
    if unknown:
        print("\n Unknown JSON schema samples (file -> keys):")
        for fn, keys in unknown:
            print(" -", fn, "->", keys)
        print("If you see this, send ONE json file content here and I will adapt the converter.")


In [7]:
# output YOLO txt labels directly into YOLO expected folders:
OUT_TRAIN = DATASET_DIR / "labels" / "train"
OUT_VAL   = DATASET_DIR / "labels" / "val"

convert_folder(JSON_TRAIN, IMAGES_TRAIN, OUT_TRAIN)
convert_folder(JSON_VAL, IMAGES_VAL, OUT_VAL)

print("Train txt labels:", len(list(OUT_TRAIN.glob("*.txt"))))
print("Val txt labels  :", len(list(OUT_VAL.glob("*.txt"))))


No json files in: C:\Users\HashTag\Desktop\VehicleClassificationYOLO\data\vehicles\labels\train
No json files in: C:\Users\HashTag\Desktop\VehicleClassificationYOLO\data\vehicles\labels\val
Train txt labels: 0
Val txt labels  : 0


In [8]:
import yaml

DATA_YAML = DATASET_DIR / "data_yolo.yaml"

data_cfg = {
    "train": str(IMAGES_TRAIN).replace("\\", "/"),
    "val":   str(IMAGES_VAL).replace("\\", "/"),
    "nc": len(CLASS_NAMES),
    "names": {i:n for i,n in enumerate(CLASS_NAMES)}
}

with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

print("Saved:", DATA_YAML)
print(data_cfg)


Saved: C:\Users\HashTag\Desktop\VehicleClassificationYOLO\data\vehicles\data_yolo.yaml
{'train': 'C:/Users/HashTag/Desktop/VehicleClassificationYOLO/data/vehicles/images/train', 'val': 'C:/Users/HashTag/Desktop/VehicleClassificationYOLO/data/vehicles/images/val', 'nc': 4, 'names': {0: 'car', 1: 'bus', 2: 'truck', 3: 'motorcycle'}}


In [9]:
from collections import Counter

def count_instances(lbl_dir: Path):
    total = 0
    cls = Counter()
    for p in lbl_dir.glob("*.txt"):
        txt = p.read_text().strip()
        for line in txt.splitlines():
            parts = line.split()
            if len(parts) == 5:
                total += 1
                cls[int(float(parts[0]))] += 1
    return total, cls

n_train, c_train = count_instances(OUT_TRAIN)
n_val, c_val = count_instances(OUT_VAL)

print("Train instances:", n_train, dict(c_train))
print("Val instances  :", n_val, dict(c_val))


Train instances: 0 {}
Val instances  : 0 {}


In [10]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=3,
    imgsz=640,
    batch=8,          # reduce if needed (4/2)
    project=str(PROJECT_DIR / "runs"),
    name="vehicles_yolov8",
    patience=10,
    verbose=True
)


New https://pypi.org/project/ultralytics/8.4.13 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.5  Python-3.11.14 torch-2.9.1+cpu CPU (13th Gen Intel Core i5-13450HX)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\HashTag\Desktop\VehicleClassificationYOLO\data\vehicles\data_yolo.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0,

c:\Users\HashTag\miniconda3\envs\ml\Lib\site-packages\ultralytics\utils\metrics.py:837: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
c:\Users\HashTag\miniconda3\envs\ml\Lib\site-packages\numpy\_core\_methods.py:137: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
        2/3         0G          0       27.4          0          0        640: 100% ━━━━━━━━━━━━ 263/263 2.5s/it 10:49<1.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 1.3it/s 44.2s0.9ss
                   all        900          0          0          0          0          0
WARNING no labels found in detect set, cannot compute metrics without labels

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


c:\Users\HashTag\miniconda3\envs\ml\Lib\site-packages\ultralytics\utils\metrics.py:837: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
c:\Users\HashTag\miniconda3\envs\ml\Lib\site-packages\numpy\_core\_methods.py:137: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


        3/3         0G          0      13.86          0          0        640: 100% ━━━━━━━━━━━━ 263/263 1.9s/it 8:23<0.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 1.7it/s 33.8s0.6ss
                   all        900          0          0          0          0          0
WARNING no labels found in detect set, cannot compute metrics without labels

3 epochs completed in 0.571 hours.


c:\Users\HashTag\miniconda3\envs\ml\Lib\site-packages\ultralytics\utils\metrics.py:837: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
c:\Users\HashTag\miniconda3\envs\ml\Lib\site-packages\numpy\_core\_methods.py:137: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


Optimizer stripped from C:\Users\HashTag\Desktop\VehicleClassificationYOLO\runs\vehicles_yolov810\weights\last.pt, 6.2MB
Optimizer stripped from C:\Users\HashTag\Desktop\VehicleClassificationYOLO\runs\vehicles_yolov810\weights\best.pt, 6.2MB

Validating C:\Users\HashTag\Desktop\VehicleClassificationYOLO\runs\vehicles_yolov810\weights\best.pt...
Ultralytics 8.4.5  Python-3.11.14 torch-2.9.1+cpu CPU (13th Gen Intel Core i5-13450HX)
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 1.9it/s 29.6s0.6ss


c:\Users\HashTag\miniconda3\envs\ml\Lib\site-packages\ultralytics\utils\metrics.py:655: RuntimeWarning: Mean of empty slice.
  ax.plot(px, py.mean(1), linewidth=3, color="blue", label=f"all classes {ap[:, 0].mean():.3f} mAP@0.5")
c:\Users\HashTag\miniconda3\envs\ml\Lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\HashTag\miniconda3\envs\ml\Lib\site-packages\ultralytics\utils\metrics.py:701: RuntimeWarning: Mean of empty slice.
  y = smooth(py.mean(0), 0.1)
c:\Users\HashTag\miniconda3\envs\ml\Lib\site-packages\numpy\_core\_methods.py:137: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
c:\Users\HashTag\miniconda3\envs\ml\Lib\site-packages\ultralytics\utils\metrics.py:701: RuntimeWarning: Mean of empty slice.
  y = smooth(py.mean(0), 0.1)
c:\Users\HashTag\miniconda3\envs\ml\Lib\site-packages\numpy\_core\_methods.py:137: RuntimeWarning: invalid value encounter

                   all        900          0          0          0          0          0
WARNING no labels found in detect set, cannot compute metrics without labels
Speed: 0.5ms preprocess, 27.9ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to C:\Users\HashTag\Desktop\VehicleClassificationYOLO\runs\vehicles_yolov810


In [15]:
from ultralytics import YOLO

weights_dir = PROJECT_DIR / "runs" / "vehicles_yolov8" / "weights"
best_weights = weights_dir / "best.pt"
last_weights = weights_dir / "last.pt"

if best_weights.exists():
    weights_path = best_weights
elif last_weights.exists():
    weights_path = last_weights
else:
    print(f"No trained weights found in {weights_dir}. Falling back to yolov8n.pt.")
    weights_path = "yolov8n.pt"

model = YOLO(str(weights_path))

IMAGE_PATH = r"C:/Users/HashTag/Desktop/input.jpg" 

model.predict(
    source=IMAGE_PATH,
    imgsz=640,
    conf=0.25,
    save=True,
    project=str(PROJECT_DIR / "runs"),
    name="predict_image"
)

print("Saved to:", PROJECT_DIR / "runs" / "predict_image")


No trained weights found in C:\Users\HashTag\Desktop\VehicleClassificationYOLO\runs\vehicles_yolov8\weights. Falling back to yolov8n.pt.

image 1/1 C:\Users\HashTag\Desktop\input.jpg: 384x640 24 cars, 1 bus, 4 trucks, 27.2ms
Speed: 1.5ms preprocess, 27.2ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)
Results saved to C:\Users\HashTag\Desktop\VehicleClassificationYOLO\runs\predict_image3
Saved to: C:\Users\HashTag\Desktop\VehicleClassificationYOLO\runs\predict_image
